<a href="https://colab.research.google.com/github/darioLabrador/TestRepo/blob/main/(FO)Training_No_Colour_.ipynb" target="_parent"><img src="https://colab.research.google.com/assets/colab-badge.svg" alt="Open In Colab"/></a>

In [1]:
!pip install torch
!pip install numpy
!pip install open_clip-torch
!pip install Pillow
!pip install scikit-learn
!pip install torch pandas

   ━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━ 1.5/1.5 MB 19.6 MB/s eta 0:00:00
   ━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━ 44.8/44.8 kB 4.6 MB/s eta 0:00:00


In [2]:
from google.colab import drive
drive.mount('/content/drive')

Mounted at /content/drive


In [3]:
import torch
import numpy as np
import random
import open_clip
from PIL import Image
import pandas as pd
import glob

def set_seed(seed=42):
    torch.manual_seed(seed)
    torch.cuda.manual_seed_all(seed)
    np.random.seed(seed)
    random.seed(seed)
    torch.backends.cudnn.deterministic = True
    torch.backends.cudnn.benchmark = False

set_seed(42)

# Setup Model
model_id = 'hf-hub:microsoft/BiomedCLIP-PubMedBERT_256-vit_base_patch16_224'
device = "cuda" if torch.cuda.is_available() else "cpu"
print(f"Loading BiomedCLIP model on {device}...")

model, _, preprocess_val = open_clip.create_model_and_transforms(model_id)
tokenizer = open_clip.get_tokenizer(model_id)
model.eval()
model.to(device)

# Define Label Sets
colour_labels = [
    "a T wave with a high amount of blue/green",
    "a T wave with a high amount of orange/red",
]

morphology_label_sets = {
    "No Label": ["Normal", "Abnormal"],
    "L1": ["Normal T wave", "Abnormal T wave"],
    "L2": ["Normal", "Flat", "Large", "Inverted", "Biphasic", "Notched"],
    "L3": [
        "Normal T wave",
        "Abnormal T wave with flat amplitude",
        "Abnormal large T wave, taller than initial signal",
        '''Abnormal large T wave, broader and larger than
         initial signal''',
        "Abnormal T wave with a minor inversion",
        "Abnormal T wave with a deep inversion",
        "Abnormal T wave with a giant inversion",
        "Abnormal Biphasic T wave(Type A)",
        "Abnormal Biphasic T wave(Type B)",
        "Abnormal Bifid or Notched T wave"
    ]
}
desired_label_set_order = ["No Label", "L1", "L2", "L3"]

# Pre-encode colour features
text_c = tokenizer(colour_labels).to(device)
with torch.no_grad():
    text_features_c = model.encode_text(text_c)
    text_features_c /= text_features_c.norm(dim=-1, keepdim=True)

# Helper Function for Inference
def evaluate_image(image_path, text_features_m, text_features_c, morphology_labels, alpha=0.5):
    raw_image = Image.open(image_path).convert('RGB')
    image = preprocess_val(raw_image).unsqueeze(0).to(device)

    with torch.no_grad():
        image_features = model.encode_image(image)
        image_features /= image_features.norm(dim=-1, keepdim=True)
        logit_scale = model.logit_scale.exp()

        logits_per_image_m = logit_scale * image_features @ text_features_m.t()
        logits_per_image_c = logit_scale * image_features @ text_features_c.t()

        probs_m = logits_per_image_m.softmax(dim=-1).cpu().numpy()[0]
        probs_c = logits_per_image_c.softmax(dim=-1).cpu().numpy()[0]

        p_blue_green = probs_c[0]
        p_orange_red = probs_c[1]

        weight_blue_green = (p_blue_green * alpha) + (1.0 - alpha)
        weight_orange_red = (p_orange_red * alpha) + (1.0 - alpha)

        mixed_probs_m = []
        for i, label in enumerate(morphology_labels):
            weight = weight_blue_green if "Normal" in label else weight_orange_red
            mixed_probs_m.append(probs_m[i] * weight)

        mixed_probs_m = np.array(mixed_probs_m)
        mixed_probs_m /= mixed_probs_m.sum()

        none_probs = np.ones(len(morphology_labels)) / len(morphology_labels)

    return probs_m, probs_c, mixed_probs_m, none_probs

Loading BiomedCLIP model on cuda...


/usr/local/lib/python3.12/dist-packages/huggingface_hub/utils/_auth.py:93: UserWarning: 
The secret `HF_TOKEN` does not exist in your Colab secrets.
To authenticate with the Hugging Face Hub, create a token in your settings tab (https://huggingface.co/settings/tokens), set it as secret in your Google Colab and restart your session.
You will be able to reuse this secret in all of your notebooks.
Please note that authentication is recommended but still optional to access public models or datasets.
  warnings.warn(


open_clip_config.json:   0%|          | 0.00/707 [00:00<?, ?B/s]

open_clip_pytorch_model.bin:   0%|          | 0.00/784M [00:00<?, ?B/s]

config.json:   0%|          | 0.00/385 [00:00<?, ?B/s]

tokenizer_config.json:   0%|          | 0.00/28.0 [00:00<?, ?B/s]

vocab.txt: 0.00B [00:00, ?B/s]

In [4]:
import glob
from PIL import Image
import numpy as np
import torch
from sklearn.linear_model import LogisticRegression
from sklearn.metrics import classification_report

print("Loading training data...")

# Define training paths
train_high_risk_dir = "/content/drive/MyDrive/Training Data(144)/no_colour_high_risk_subset/*.png"
train_no_risk_dir = "/content/drive/MyDrive/Training Data(144)/no_colour_no_risk_subset/*.png"

train_paths_abnormal = sorted(glob.glob(train_high_risk_dir))
train_paths_normal = sorted(glob.glob(train_no_risk_dir))

train_paths = train_paths_abnormal + train_paths_normal

# Labels: 1 for Abnormal, 0 for Normal
train_labels = [1] * len(train_paths_abnormal) + [0] * len(train_paths_normal)

print(f"Found {len(train_paths_abnormal)} Abnormal and {len(train_paths_normal)} Normal training images.")

def extract_features(paths):
    features = []
    model.eval()
    with torch.no_grad():
        for path in paths:
            try:
                raw_image = Image.open(path).convert('RGB')
                image = preprocess_val(raw_image).unsqueeze(0).to(device)
                image_features = model.encode_image(image)
                image_features /= image_features.norm(dim=-1, keepdim=True)
                features.append(image_features.cpu().numpy()[0])
            except Exception as e:
                print(f"Error processing {path}: {e}")
                # Append a zero vector if image fails to load to maintain alignment, though ideally it should be removed
                features.append(np.zeros(model.visual.output_dim))
    return np.array(features)

print("Extracting embeddings using BiomedCLIP... (This may take a moment)")
X_train = extract_features(train_paths)
y_train = np.array(train_labels)

# Train a simple classifier
print("Training Logistic Regression classifier...")
classifier = LogisticRegression(random_state=42, max_iter=1000)
classifier.fit(X_train, y_train)

# Evaluate on training data to verify it learned
train_preds = classifier.predict(X_train)
print("\nTraining Classification Report:")
print(classification_report(y_train, train_preds, target_names=['Normal', 'Abnormal']))


Loading training data...
Found 144 Abnormal and 144 Normal training images.
Extracting embeddings using BiomedCLIP... (This may take a moment)
Training Logistic Regression classifier...

Training Classification Report:
              precision    recall  f1-score   support

      Normal       0.93      0.96      0.94       144
    Abnormal       0.96      0.92      0.94       144

    accuracy                           0.94       288
   macro avg       0.94      0.94      0.94       288
weighted avg       0.94      0.94      0.94       288



In [5]:
import glob
import numpy as np
from sklearn.metrics import classification_report

print("Extracting embeddings for the Test Data(37) using BiomedCLIP")
# Use both abnormal and normal test images
test_high_risk_dir = "/content/drive/MyDrive/Test Data(37)/no_colour_high_risk_subset/*.png"
test_no_risk_dir = "/content/drive/MyDrive/Test Data(37)/no_colour_no_risk_subset/*.png"

test_paths_abnormal = sorted(glob.glob(test_high_risk_dir))
test_paths_normal = sorted(glob.glob(test_no_risk_dir))

all_test_paths = test_paths_abnormal + test_paths_normal

# Convert to binary (1 for Abnormal, 0 for Normal) to match training
y_test_binary = np.array([1] * len(test_paths_abnormal) + [0] * len(test_paths_normal))

X_test = extract_features(all_test_paths)

print("Evaluating Logistic Regression classifier on Test Data(37)...")
test_preds = classifier.predict(X_test)

print("\nTest Classification Report (Linear Probe):")
print(classification_report(y_test_binary, test_preds, target_names=['Normal', 'Abnormal']))

Extracting embeddings for the Test Data(37) using BiomedCLIP
Evaluating Logistic Regression classifier on Test Data(37)...

Test Classification Report (Linear Probe):
              precision    recall  f1-score   support

      Normal       0.67      0.05      0.10        37
    Abnormal       0.51      0.97      0.67        37

    accuracy                           0.51        74
   macro avg       0.59      0.51      0.38        74
weighted avg       0.59      0.51      0.38        74



In [6]:
import glob
import numpy as np

# Define real image paths
high_risk_dir = "/content/drive/MyDrive/Heartbeat_noColour/Heartbeat_noColour/At risk/*.png"
no_risk_dir = "/content/drive/MyDrive/Heartbeat_noColour/Heartbeat_noColour/No risk/*.png"

# Retrieve and sort .png files
image_paths_abnormal = np.array(sorted(glob.glob(high_risk_dir)))
image_paths_normal = np.array(sorted(glob.glob(no_risk_dir)))

# Create ground truth labels
ground_truth_abnormal = np.array(['Abnormal'] * len(image_paths_abnormal))
ground_truth_normal = np.array(['Normal'] * len(image_paths_normal))

# Combine arrays
image_paths = np.concatenate((image_paths_abnormal, image_paths_normal))
ground_truth = np.concatenate((ground_truth_abnormal, ground_truth_normal))

# Print counts
print(f"Found {len(image_paths_abnormal)} high-risk (abnormal) images.")
print(f"Found {len(image_paths_normal)} no-risk (normal) images.")
print(f"Total combined images: {len(image_paths)}")
print(f"Total ground truth labels: {len(ground_truth)}")

Found 181 high-risk (abnormal) images.
Found 4573 no-risk (normal) images.
Total combined images: 4754
Total ground truth labels: 4754


In [7]:
import pandas as pd
import numpy as np
import torch
import os

real_inference_results_colour = []

print("Starting inference loop including Colour Strategy...")
for label_set_name in desired_label_set_order:
    morph_labels = morphology_label_sets[label_set_name]

    # Pre-encode text features for the current morphology labels
    text_m = tokenizer(morph_labels).to(device)
    with torch.no_grad():
        text_features_m = model.encode_text(text_m)
        text_features_m /= text_features_m.norm(dim=-1, keepdim=True)

    # Iterate through all real image paths
    for path, true_label in zip(image_paths, ground_truth):
        try:
            # Evaluate image
            p_morph, p_col, p_both, p_none = evaluate_image(
                path, text_features_m, text_features_c, morph_labels, alpha=0.5
            )

            # Extract predicted labels
            pred_morph = morph_labels[np.argmax(p_morph)]
            pred_both = morph_labels[np.argmax(p_both)]
            pred_none = morph_labels[np.argmax(p_none)]

            # Colour Strategy mapping: blue/green (index 0) -> Normal, orange/red (index 1) -> Abnormal
            pred_col = 'Normal' if np.argmax(p_col) == 0 else 'Abnormal'

            # Extract file name from path
            file_name = os.path.basename(path)

            # No Colour + Morphology Strategy
            real_inference_results_colour.append({
                'File_Name': file_name,
                'True_Label': true_label,
                'Predicted_Label': pred_morph,
                'Strategy': 'No Colour + Morphology Context',
                'Label_Set': label_set_name
            })

            # Colour + Morphology Strategy
            real_inference_results_colour.append({
                'File_Name': file_name,
                'True_Label': true_label,
                'Predicted_Label': pred_both,
                'Strategy': 'Colour + Morphology Context',
                'Label_Set': label_set_name
            })

            # No Colour + No Morphology Strategy
            real_inference_results_colour.append({
                'File_Name': file_name,
                'True_Label': true_label,
                'Predicted_Label': pred_none,
                'Strategy': 'No Colour + No Morphology Context',
                'Label_Set': label_set_name
            })

            # Colour + No Morphology Strategy
            real_inference_results_colour.append({
                'File_Name': file_name,
                'True_Label': true_label,
                'Predicted_Label': pred_col,
                'Strategy': 'Colour + No Morphology Context',
                'Label_Set': label_set_name
            })

        except Exception as e:
            print(f"Error processing {path}: {e}")

# Convert results to DataFrame
df_real_inference_colour = pd.DataFrame(real_inference_results_colour)
print("\nInference including Colour Strategy complete! First few results:")
display(df_real_inference_colour.head(10))

Starting inference loop including Colour Strategy...

Inference including Colour Strategy complete! First few results:


,File_Name,True_Label,Predicted_Label,Strategy,Label_Set
0,1415_Heartbeat_1.png,Abnormal,Abnormal,No Colour + Morphology Context,No Label
1,1415_Heartbeat_1.png,Abnormal,Abnormal,Colour + Morphology Context,No Label
2,1415_Heartbeat_1.png,Abnormal,Normal,No Colour + No Morphology Context,No Label
3,1415_Heartbeat_1.png,Abnormal,Abnormal,Colour + No Morphology Context,No Label
4,1426_Heartbeat_1.png,Abnormal,Abnormal,No Colour + Morphology Context,No Label
5,1426_Heartbeat_1.png,Abnormal,Abnormal,Colour + Morphology Context,No Label
6,1426_Heartbeat_1.png,Abnormal,Normal,No Colour + No Morphology Context,No Label
7,1426_Heartbeat_1.png,Abnormal,Abnormal,Colour + No Morphology Context,No Label
8,1441_Heartbeat_1.png,Abnormal,Abnormal,No Colour + Morphology Context,No Label
9,1441_Heartbeat_1.png,Abnormal,Abnormal,Colour + Morphology Context,No Label


In [8]:
import math
from sklearn.metrics import roc_curve, roc_auc_score, confusion_matrix
import matplotlib.pyplot as plt

# Map detailed predictions to binary
def map_to_binary(label):
    if 'normal' in label.lower() and 'abnormal' not in label.lower():
        return 'Normal'
    elif 'normal' in label.lower() and label.lower().startswith('normal'):
         return 'Normal'
    else:
        return 'Abnormal'

df_real_inference_colour['Binary_Prediction'] = df_real_inference_colour['Predicted_Label'].apply(map_to_binary)

# Map to numeric binary for sklearn metrics
df_real_inference_colour['True_Binary_Num'] = (df_real_inference_colour['True_Label'] == 'Abnormal').astype(int)
df_real_inference_colour['Pred_Binary_Num'] = (df_real_inference_colour['Binary_Prediction'] == 'Abnormal').astype(int)

# Initialize metrics list
metrics_colour_results = []

# Group and calculate metrics
grouped_colour = df_real_inference_colour.groupby(['Label_Set', 'Strategy'])

plt.figure(figsize=(12, 8))

for (label_set, strategy), group in grouped_colour:
    y_true = group['True_Binary_Num']
    y_pred = group['Pred_Binary_Num']

    TP = len(group[(group['True_Label'] == 'Abnormal') & (group['Binary_Prediction'] == 'Abnormal')])
    FN = len(group[(group['True_Label'] == 'Abnormal') & (group['Binary_Prediction'] == 'Normal')])
    TN = len(group[(group['True_Label'] == 'Normal') & (group['Binary_Prediction'] == 'Normal')])
    FP = len(group[(group['True_Label'] == 'Normal') & (group['Binary_Prediction'] == 'Abnormal')])

    recall = TP / (TP + FN) if (TP + FN) > 0 else 0.0
    specificity = TN / (TN + FP) if (TN + FP) > 0 else 0.0

    # New Metrics
    ppv = TP / (TP + FP) if (TP + FP) > 0 else 0.0
    f1 = 2 * (ppv * recall) / (ppv + recall) if (ppv + recall) > 0 else 0.0

    denom = math.sqrt((TP + FP) * (TP + FN) * (TN + FP) * (TN + FN))
    mcc = ((TP * TN) - (FP * FN)) / denom if denom > 0 else 0.0

    # ROC AUC (using discrete predictions since raw probabilities aren't stored in df)
    try:
        roc_auc = roc_auc_score(y_true, y_pred)
    except ValueError:
        roc_auc = 0.0

    # Append results
    metrics_colour_results.append({
        'Label_Set': label_set,
        'Strategy': strategy,
        'TP': TP,
        'FN': FN,
        'TN': TN,
        'FP': FP,
        'Sensitivity': recall,
        'Specificity': specificity,
        'PPV': ppv,
        'F1_Score': f1,
        'MCC': mcc,
        'ROC_AUC': roc_auc
    })

# Linear Probing
if 'y_test_binary' in globals() and 'test_preds' in globals():
    tn, fp, fn, tp = confusion_matrix(y_test_binary, test_preds).ravel()
    recall_lp = tp / (tp + fn) if (tp + fn) > 0 else 0.0
    specificity_lp = tn / (tn + fp) if (tn + fp) > 0 else 0.0
    ppv_lp = tp / (tp + fp) if (tp + fp) > 0 else 0.0
    f1_lp = 2 * (ppv_lp * recall_lp) / (ppv_lp + recall_lp) if (ppv_lp + recall_lp) > 0 else 0.0

    denom_lp = math.sqrt((tp + fp) * (tp + fn) * (tn + fp) * (tn + fn))
    mcc_lp = ((tp * tn) - (fp * fn)) / denom_lp if denom_lp > 0 else 0.0
    roc_auc_lp = roc_auc_score(y_test_binary, test_preds)

    fpr_lp, tpr_lp, _ = roc_curve(y_test_binary, test_preds)

    metrics_colour_results.append({
        'Label_Set': 'N/A (Trained)',
        'Strategy': 'Linear Probe',
        'TP': tp,
        'FN': fn,
        'TN': tn,
        'FP': fp,
        'Sensitivity': recall_lp,
        'Specificity': specificity_lp,
        'PPV': ppv_lp,
        'F1_Score': f1_lp,
        'MCC': mcc_lp,
        'ROC_AUC': roc_auc_lp
    })

# Output Dataframe
df_metrics_colour = pd.DataFrame(metrics_colour_results)
print("Final Metrics Including Colour Strategy, Linear Probe, and Additional Metrics:")
display(df_metrics_colour)


Final Metrics Including Colour Strategy, Linear Probe, and Additional Metrics:


,Label_Set,Strategy,TP,FN,TN,FP,Sensitivity,Specificity,PPV,F1_Score,MCC,ROC_AUC
0,L1,Colour + Morphology Context,175,6,140,4433,0.966851,0.030614,0.037977,0.073084,-0.002811,0.498733
1,L1,Colour + No Morphology Context,181,0,0,4573,1.000000,0.000000,0.038073,0.073354,0.000000,0.500000
2,L1,No Colour + Morphology Context,167,14,601,3972,0.922652,0.131424,0.040348,0.077315,0.030836,0.527038
3,L1,No Colour + No Morphology Context,0,181,4573,0,0.000000,1.000000,0.000000,0.000000,0.000000,0.500000
4,L2,Colour + Morphology Context,180,1,26,4547,0.994475,0.005686,0.038079,0.073350,0.000409,0.500080
5,L2,Colour + No Morphology Context,181,0,0,4573,1.000000,0.000000,0.038073,0.073354,0.000000,0.500000
6,L2,No Colour + Morphology Context,180,1,65,4508,0.994475,0.014214,0.038396,0.073937,0.014212,0.504345
7,L2,No Colour + No Morphology Context,0,181,4573,0,0.000000,1.000000,0.000000,0.000000,0.000000,0.500000
8,L3,Colour + Morphology Context,144,37,1119,3454,0.795580,0.244697,0.040022,0.076211,0.017968,0.520139
9,L3,Colour + No Morphology Context,181,0,0,4573,1.000000,0.000000,0.038073,0.073354,0.000000,0.500000


<Figure size 1200x800 with 0 Axes>

In [9]:
import numpy as np
import pandas as pd
from sklearn.model_selection import StratifiedKFold
from sklearn.linear_model import LogisticRegression
from sklearn.metrics import roc_auc_score, confusion_matrix
import torch
from PIL import Image
import math

# Prepare binary labels for real data (1 = Abnormal, 0 = Normal)
y_true_numeric = np.array([1 if t == 'Abnormal' else 0 for t in ground_truth])

print("Pre-extracting features for all real images...")
# Pre-extract image embeddings and zero-shot probabilities
image_embeddings_list = []
p_morph_dict = {ls: [] for ls in desired_label_set_order}
p_col_dict = {ls: [] for ls in desired_label_set_order}
p_both_dict = {ls: [] for ls in desired_label_set_order}
p_none_dict = {ls: [] for ls in desired_label_set_order}

model.eval()
with torch.no_grad():
    for path in image_paths:
        try:
            raw_image = Image.open(path).convert('RGB')
            image = preprocess_val(raw_image).unsqueeze(0).to(device)
            img_feat = model.encode_image(image)
            img_feat /= img_feat.norm(dim=-1, keepdim=True)
            img_feat_np = img_feat.cpu().numpy()[0]
        except Exception as e:
            print(f"Error loading {path}: {e}")
            img_feat_np = np.zeros(model.visual.output_dim)

        image_embeddings_list.append(img_feat_np)

        # Get text probabilities for each label set
        for label_set_name in desired_label_set_order:
            morph_labels = morphology_label_sets[label_set_name]
            text_m = tokenizer(morph_labels).to(device)
            text_features_m = model.encode_text(text_m)
            text_features_m /= text_features_m.norm(dim=-1, keepdim=True)

            p_morph, p_col, p_both, p_none = evaluate_image(
                path, text_features_m, text_features_c, morph_labels, alpha=0.5
            )

            p_morph_dict[label_set_name].append(p_morph)
            p_col_dict[label_set_name].append(p_col)
            p_both_dict[label_set_name].append(p_both)
            p_none_dict[label_set_name].append(p_none)

X_emb = np.array(image_embeddings_list)

print("Extraction complete. Running 5-Fold Cross Validation...")
# Run Cross-Validation for each strategy and label set
skf = StratifiedKFold(n_splits=5, shuffle=True, random_state=42)
concat_cv_results = []

# Tracking individual predictions for distribution analysis
cv_predictions_list = []

for label_set_name in desired_label_set_order:
    features_strategies = {
        'No Colour + Morphology': np.array(p_morph_dict[label_set_name]),
        'Colour + No Morphology': np.array(p_col_dict[label_set_name]),
        'Colour + Morphology': np.array(p_both_dict[label_set_name]),
        'No Colour + No Morphology': np.array(p_none_dict[label_set_name])
    }

    for strategy_name, strategy_probs in features_strategies.items():
        # Concatenate image embeddings with strategy probabilities
        X_concat = np.hstack((X_emb, strategy_probs))

        fold_y_true = []
        fold_y_pred = []
        fold_y_prob = []

        for train_idx, val_idx in skf.split(X_concat, y_true_numeric):
            X_train_cv, X_val_cv = X_concat[train_idx], X_concat[val_idx]
            y_train_cv, y_val_cv = y_true_numeric[train_idx], y_true_numeric[val_idx]

            clf = LogisticRegression(max_iter=1000, random_state=42)
            clf.fit(X_train_cv, y_train_cv)

            preds = clf.predict(X_val_cv)
            probs = clf.predict_proba(X_val_cv)[:, 1]

            fold_y_true.extend(y_val_cv)
            fold_y_pred.extend(preds)
            fold_y_prob.extend(probs)

        # Track predictions for distribution analysis
        for p in fold_y_pred:
            cv_predictions_list.append({
                'Label_Set': label_set_name,
                'Strategy': strategy_name,
                'Binary_Prediction': 'Abnormal' if p == 1 else 'Normal'
            })

        # Calculate metrics over all out-of-fold predictions
        tn, fp, fn, tp = confusion_matrix(fold_y_true, fold_y_pred).ravel()
        recall = tp / (tp + fn) if (tp + fn) > 0 else 0.0
        specificity = tn / (tn + fp) if (tn + fp) > 0 else 0.0
        auc = roc_auc_score(fold_y_true, fold_y_prob)

        ppv = tp / (tp + fp) if (tp + fp) > 0 else 0.0
        f1 = 2 * (ppv * recall) / (ppv + recall) if (ppv + recall) > 0 else 0.0
        denom = math.sqrt((tp + fp) * (tp + fn) * (tn + fp) * (tn + fn))
        mcc = ((tp * tn) - (fp * fn)) / denom if denom > 0 else 0.0

        concat_cv_results.append({
            'Label_Set': label_set_name,
            'Strategy': strategy_name,
            'TP': tp,
            'FN': fn,
            'TN': tn,
            'FP': fp,
            'Sensitivity (Recall)': recall,
            'Specificity': specificity,
            'PPV': ppv,
            'F1_Score': f1,
            'MCC': mcc,
            'ROC_AUC': auc
        })

df_concat_cv = pd.DataFrame(concat_cv_results)
display(df_concat_cv)

Pre-extracting features for all real images...
Extraction complete. Running 5-Fold Cross Validation...


,Label_Set,Strategy,TP,FN,TN,FP,Sensitivity (Recall),Specificity,PPV,F1_Score,MCC,ROC_AUC
0,No Label,No Colour + Morphology,0,181,4573,0,0.0,1.0,0.0,0.0,0.0,0.840464
1,No Label,Colour + No Morphology,0,181,4573,0,0.0,1.0,0.0,0.0,0.0,0.846273
2,No Label,Colour + Morphology,0,181,4573,0,0.0,1.0,0.0,0.0,0.0,0.842383
3,No Label,No Colour + No Morphology,0,181,4573,0,0.0,1.0,0.0,0.0,0.0,0.843786
4,L1,No Colour + Morphology,0,181,4573,0,0.0,1.0,0.0,0.0,0.0,0.820693
5,L1,Colour + No Morphology,0,181,4573,0,0.0,1.0,0.0,0.0,0.0,0.846273
6,L1,Colour + Morphology,0,181,4573,0,0.0,1.0,0.0,0.0,0.0,0.826493
7,L1,No Colour + No Morphology,0,181,4573,0,0.0,1.0,0.0,0.0,0.0,0.843786
8,L2,No Colour + Morphology,0,181,4573,0,0.0,1.0,0.0,0.0,0.0,0.810638
9,L2,Colour + No Morphology,0,181,4573,0,0.0,1.0,0.0,0.0,0.0,0.846273


In [10]:
from sklearn.model_selection import StratifiedKFold
from sklearn.linear_model import LogisticRegression
from sklearn.metrics import roc_auc_score, confusion_matrix
import pandas as pd
import numpy as np
import math

print("Running 5-Fold Cross Validation to output per-fold metrics...")
skf = StratifiedKFold(n_splits=5, shuffle=True, random_state=42)
fold_cv_results = []

for label_set_name in desired_label_set_order:
    features_strategies = {
        'No Colour + Morphology': np.array(p_morph_dict[label_set_name]),
        'Colour + No Morphology': np.array(p_col_dict[label_set_name]),
        'Colour + Morphology': np.array(p_both_dict[label_set_name]),
        'No Colour + No Morphology': np.array(p_none_dict[label_set_name])
    }

    for strategy_name, strategy_probs in features_strategies.items():
        # Concatenate image embeddings with strategy probabilities
        X_concat = np.hstack((X_emb, strategy_probs))

        fold_idx = 1
        for train_idx, val_idx in skf.split(X_concat, y_true_numeric):
            X_train_cv, X_val_cv = X_concat[train_idx], X_concat[val_idx]
            y_train_cv, y_val_cv = y_true_numeric[train_idx], y_true_numeric[val_idx]

            clf = LogisticRegression(max_iter=1000, random_state=42, class_weight='balanced')
            clf.fit(X_train_cv, y_train_cv)

            preds = clf.predict(X_val_cv)
            probs = clf.predict_proba(X_val_cv)[:, 1]

            # Calculate metrics for the current fold
            tn, fp, fn, tp = confusion_matrix(y_val_cv, preds).ravel()
            recall = tp / (tp + fn) if (tp + fn) > 0 else 0.0
            specificity = tn / (tn + fp) if (tn + fp) > 0 else 0.0

            try:
                auc = roc_auc_score(y_val_cv, probs)
            except ValueError:
                auc = 0.0

            ppv = tp / (tp + fp) if (tp + fp) > 0 else 0.0
            f1 = 2 * (ppv * recall) / (ppv + recall) if (ppv + recall) > 0 else 0.0
            denom = math.sqrt((tp + fp) * (tp + fn) * (tn + fp) * (tn + fn))
            mcc = ((tp * tn) - (fp * fn)) / denom if denom > 0 else 0.0

            fold_cv_results.append({
                'Label_Set': label_set_name,
                'Strategy': strategy_name,
                'Fold': fold_idx,
                'TP': tp,
                'FN': fn,
                'TN': tn,
                'FP': fp,
                'Sensitivity': recall,
                'Specificity': specificity,
                'PPV': ppv,
                'F1_Score': f1,
                'MCC': mcc,
                'ROC_AUC': auc
            })
            fold_idx += 1

df_fold_cv = pd.DataFrame(fold_cv_results)

# Desired columns to display
display_columns = [
    'Label_Set', 'Strategy', 'TP', 'FN', 'TN', 'FP',
    'Sensitivity', 'Specificity', 'PPV', 'F1_Score', 'MCC', 'ROC_AUC'
]

# Display a separate table for each fold
for fold in range(1, 6):
    print(f"\n{'='*60}")
    print(f"Metrics for Fold {fold}")
    print(f"{'='*60}")
    df_fold = df_fold_cv[df_fold_cv['Fold'] == fold][display_columns].copy()
    display(df_fold)


Running 5-Fold Cross Validation to output per-fold metrics...

Metrics for Fold 1


,Label_Set,Strategy,TP,FN,TN,FP,Sensitivity,Specificity,PPV,F1_Score,MCC,ROC_AUC
0,No Label,No Colour + Morphology,30,7,767,147,0.810811,0.839168,0.169492,0.280374,0.322936,0.921225
5,No Label,Colour + No Morphology,31,6,769,145,0.837838,0.841357,0.176136,0.291080,0.338190,0.921462
10,No Label,Colour + Morphology,30,7,768,146,0.810811,0.840263,0.170455,0.281690,0.324188,0.920545
15,No Label,No Colour + No Morphology,30,7,764,150,0.810811,0.835886,0.166667,0.276498,0.319236,0.921048
20,L1,No Colour + Morphology,29,8,768,146,0.783784,0.840263,0.165714,0.273585,0.311416,0.920634
25,L1,Colour + No Morphology,31,6,769,145,0.837838,0.841357,0.176136,0.291080,0.338190,0.921462
30,L1,Colour + Morphology,28,9,769,145,0.756757,0.841357,0.161850,0.266667,0.299809,0.919008
35,L1,No Colour + No Morphology,30,7,764,150,0.810811,0.835886,0.166667,0.276498,0.319236,0.921048
40,L2,No Colour + Morphology,30,7,762,152,0.810811,0.833698,0.164835,0.273973,0.316814,0.923443
45,L2,Colour + No Morphology,31,6,769,145,0.837838,0.841357,0.176136,0.291080,0.338190,0.921462



Metrics for Fold 2


,Label_Set,Strategy,TP,FN,TN,FP,Sensitivity,Specificity,PPV,F1_Score,MCC,ROC_AUC
1,No Label,No Colour + Morphology,30,6,735,180,0.833333,0.803279,0.142857,0.243902,0.292899,0.872495
6,No Label,Colour + No Morphology,29,7,741,174,0.805556,0.809836,0.142857,0.242678,0.286625,0.872192
11,No Label,Colour + Morphology,30,6,736,179,0.833333,0.804372,0.143541,0.244898,0.293905,0.872435
16,No Label,No Colour + No Morphology,30,6,743,172,0.833333,0.812022,0.148515,0.252101,0.301123,0.883789
21,L1,No Colour + Morphology,30,6,738,177,0.833333,0.806557,0.144928,0.246914,0.295935,0.879083
26,L1,Colour + No Morphology,29,7,741,174,0.805556,0.809836,0.142857,0.242678,0.286625,0.872192
31,L1,Colour + Morphology,30,6,736,179,0.833333,0.804372,0.143541,0.244898,0.293905,0.878901
36,L1,No Colour + No Morphology,30,6,743,172,0.833333,0.812022,0.148515,0.252101,0.301123,0.883789
41,L2,No Colour + Morphology,30,6,735,180,0.833333,0.803279,0.142857,0.243902,0.292899,0.886764
46,L2,Colour + No Morphology,29,7,741,174,0.805556,0.809836,0.142857,0.242678,0.286625,0.872192



Metrics for Fold 3


,Label_Set,Strategy,TP,FN,TN,FP,Sensitivity,Specificity,PPV,F1_Score,MCC,ROC_AUC
2,No Label,No Colour + Morphology,29,7,759,156,0.805556,0.829508,0.156757,0.262443,0.306181,0.878689
7,No Label,Colour + No Morphology,29,7,761,154,0.805556,0.831694,0.158470,0.264840,0.308507,0.879508
12,No Label,Colour + Morphology,29,7,761,154,0.805556,0.831694,0.158470,0.264840,0.308507,0.877899
17,No Label,No Colour + No Morphology,29,7,764,151,0.805556,0.834973,0.161111,0.268519,0.312059,0.879417
22,L1,No Colour + Morphology,30,6,762,153,0.833333,0.832787,0.163934,0.273973,0.322484,0.879842
27,L1,Colour + No Morphology,29,7,761,154,0.805556,0.831694,0.158470,0.264840,0.308507,0.879508
32,L1,Colour + Morphology,30,6,762,153,0.833333,0.832787,0.163934,0.273973,0.322484,0.879660
37,L1,No Colour + No Morphology,29,7,764,151,0.805556,0.834973,0.161111,0.268519,0.312059,0.879417
42,L2,No Colour + Morphology,30,6,754,161,0.833333,0.824044,0.157068,0.264317,0.313150,0.882149
47,L2,Colour + No Morphology,29,7,761,154,0.805556,0.831694,0.158470,0.264840,0.308507,0.879508



Metrics for Fold 4


,Label_Set,Strategy,TP,FN,TN,FP,Sensitivity,Specificity,PPV,F1_Score,MCC,ROC_AUC
3,No Label,No Colour + Morphology,30,6,747,168,0.833333,0.816393,0.151515,0.256410,0.305395,0.919581
8,No Label,Colour + No Morphology,30,6,747,168,0.833333,0.816393,0.151515,0.256410,0.305395,0.918276
13,No Label,Colour + Morphology,31,5,748,167,0.861111,0.817486,0.156566,0.264957,0.318966,0.919399
18,No Label,No Colour + No Morphology,31,5,746,169,0.861111,0.815301,0.155000,0.262712,0.316766,0.919429
23,L1,No Colour + Morphology,31,5,747,168,0.861111,0.816393,0.155779,0.263830,0.317862,0.919399
28,L1,Colour + No Morphology,30,6,747,168,0.833333,0.816393,0.151515,0.256410,0.305395,0.918276
33,L1,Colour + Morphology,31,5,747,168,0.861111,0.816393,0.155779,0.263830,0.317862,0.919399
38,L1,No Colour + No Morphology,31,5,746,169,0.861111,0.815301,0.155000,0.262712,0.316766,0.919429
43,L2,No Colour + Morphology,31,5,751,164,0.861111,0.820765,0.158974,0.268398,0.322321,0.882908
48,L2,Colour + No Morphology,30,6,747,168,0.833333,0.816393,0.151515,0.256410,0.305395,0.918276



Metrics for Fold 5


,Label_Set,Strategy,TP,FN,TN,FP,Sensitivity,Specificity,PPV,F1_Score,MCC,ROC_AUC
4,No Label,No Colour + Morphology,30,6,748,166,0.833333,0.818381,0.153061,0.258621,0.307516,0.877857
9,No Label,Colour + No Morphology,30,6,742,172,0.833333,0.811816,0.148515,0.252101,0.301064,0.877857
14,No Label,Colour + Morphology,30,6,748,166,0.833333,0.818381,0.153061,0.258621,0.307516,0.877522
19,No Label,No Colour + No Morphology,30,6,751,163,0.833333,0.821663,0.155440,0.262009,0.310840,0.876884
24,L1,No Colour + Morphology,30,6,751,163,0.833333,0.821663,0.155440,0.262009,0.310840,0.877371
29,L1,Colour + No Morphology,30,6,742,172,0.833333,0.811816,0.148515,0.252101,0.301064,0.877857
34,L1,Colour + Morphology,30,6,752,162,0.833333,0.822757,0.156250,0.263158,0.311963,0.876945
39,L1,No Colour + No Morphology,30,6,751,163,0.833333,0.821663,0.155440,0.262009,0.310840,0.876884
44,L2,No Colour + Morphology,27,9,727,187,0.750000,0.795405,0.126168,0.216000,0.249286,0.851933
49,L2,Colour + No Morphology,30,6,742,172,0.833333,0.811816,0.148515,0.252101,0.301064,0.877857
